# Fiorell.IA — NO-GO Recovery Runbook

Obiettivo: diagnosticare metriche NaN, correggere stile italiano, rafforzare abstention e preparare ablation study senza cambiare architettura, base model o logica LoRA.


In [ ]:
from pathlib import Path
import sys, json

RUN_ENV = 'colab'
REPO_ROOT = Path('/content/regulatory-insight-engine') if RUN_ENV == 'colab' else Path.cwd()
ARTIFACT_ROOT = Path('/content/drive/MyDrive/fiorellia/nogo_recovery') if RUN_ENV == 'colab' else REPO_ROOT / 'artifacts/fiorellia/nogo_recovery'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))

TRAIN_JSONL = REPO_ROOT / 'fiorellia/training/supervised_v1_curated_20260421.jsonl'
BASE_CONFIG = REPO_ROOT / 'fiorellia/training/configs/config_lora_behavior_20260421.yaml'
PATCHED_JSONL = REPO_ROOT / 'fiorellia/training/supervised_v1_curated_20260421_style_abstention_patch.jsonl'
PATCHED_CONFIG = REPO_ROOT / 'fiorellia/training/configs/config_lora_behavior_20260421_style_abstention_patch.yaml'
ABLATION_ROOT = ARTIFACT_ROOT / 'ablation'

print('REPO_ROOT:', REPO_ROOT)
print('ARTIFACT_ROOT:', ARTIFACT_ROOT)


## 1 — Preflight conservativo
Verifica config, dataset e GPU prima di qualsiasi training.


In [ ]:
from fiorellia.training.fiorellia_colab_pipeline import (
    check_cuda, load_config, validate_config, require_file, write_json
)

gpu = check_cuda(require_gpu=True)
cfg = load_config(BASE_CONFIG)
resolved = validate_config(cfg, REPO_ROOT)
require_file(TRAIN_JSONL, 'training JSONL')

preflight = {'gpu': gpu, 'dataset_path': str(resolved['dataset_path']), 'output_dir': str(resolved['output_dir'])}
write_json(preflight, ARTIFACT_ROOT / '01_preflight.json')
print(json.dumps(preflight, indent=2, ensure_ascii=False))


## 2 — Debug metriche NaN
Inserisci qui il path dell'output eval JSONL/CSV più recente. Il parser produce subset counts: se un count è 0, il NaN deriva dal benchmark/mapping label.


In [ ]:
from fiorellia.training.fiorellia_colab_pipeline import read_jsonl, score_eval_rows, write_jsonl
import csv

EVAL_JSONL = ARTIFACT_ROOT / 'eval_adapter.jsonl'  # cambia se il file è altrove
EVAL_CSV = ARTIFACT_ROOT / 'comparison.csv'       # fallback CSV

def read_csv_rows(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

if EVAL_JSONL.exists():
    eval_rows = read_jsonl(EVAL_JSONL)
elif EVAL_CSV.exists():
    eval_rows = read_csv_rows(EVAL_CSV)
else:
    eval_rows = []
    print('Nessun output eval trovato in ARTIFACT_ROOT. Copia eval_adapter.jsonl o comparison.csv e riesegui questa cella.')

if eval_rows:
    scored_rows, metrics = score_eval_rows(eval_rows)
    write_json(metrics, ARTIFACT_ROOT / '02_metric_debug_summary.json')
    write_jsonl(scored_rows, ARTIFACT_ROOT / '02_scored_eval_rows.jsonl')
    print(json.dumps(metrics, indent=2, ensure_ascii=False))


## 3 — Patch stile italiano + abstention dataset
Crea un nuovo dataset SFT. L'originale non viene sovrascritto. Target abstention ratio: 40%.


In [ ]:
from fiorellia.training.fiorellia_colab_pipeline import build_style_abstention_dataset, save_config

patch_info = build_style_abstention_dataset(
    TRAIN_JSONL,
    PATCHED_JSONL,
    target_abstention_ratio=0.40,
)

patched_cfg = dict(cfg)
patched_cfg['dataset_path'] = str(PATCHED_JSONL.relative_to(REPO_ROOT))
patched_cfg['output_dir'] = 'fiorellia/training/lora/fiorellia_behavior_20260421_style_abstention_patch'
save_config(patched_cfg, PATCHED_CONFIG)

write_json({'patch_info': patch_info, 'patched_config': str(PATCHED_CONFIG)}, ARTIFACT_ROOT / '03_dataset_patch_summary.json')
print(json.dumps({'patch_info': patch_info, 'patched_config': str(PATCHED_CONFIG)}, indent=2, ensure_ascii=False))


## 4 — Training recovery
Esegui il training LoRA con config patched. Base model e LoRA restano invariati.


In [ ]:
import subprocess

TRAIN_SCRIPT = REPO_ROOT / 'fiorellia/training/train_lora_behavior_v1.py'
if not TRAIN_SCRIPT.exists():
    TRAIN_SCRIPT = REPO_ROOT / 'fiorellia/training/train_lora_behavior.py'
require_file(TRAIN_SCRIPT, 'training script')

cmd = [sys.executable, str(TRAIN_SCRIPT), '--config', str(PATCHED_CONFIG)]
print('+', ' '.join(cmd))
# Decommenta per eseguire davvero il training in Colab A100
# subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)
print('Training command prepared. Decommenta subprocess.run quando sei su runtime T4 pronto.')


## 5 — Ablation Study
Genera un dataset per ogni categoria rimossa da `category`. Se `category` manca, viene inferita in modo conservativo.


In [ ]:
from fiorellia.training.fiorellia_colab_pipeline import write_ablation_datasets, write_csv

ablation_rows = write_ablation_datasets(PATCHED_JSONL, ABLATION_ROOT)
write_csv(ablation_rows, ABLATION_ROOT / 'ablation_dataset_index.csv')
write_json({'ablation_datasets': ablation_rows}, ARTIFACT_ROOT / '05_ablation_dataset_index.json')
print(json.dumps(ablation_rows, indent=2, ensure_ascii=False))


## 6 — Quick ablation eval placeholder
Questa cella produce una metrica dataset-level immediata. Per delta performance reale, collega qui il training/eval rapido del tuo harness.


In [ ]:
from fiorellia.training.fiorellia_colab_pipeline import read_jsonl, is_abstention_training_case

quick_results = []
for row in ablation_rows:
    rows = read_jsonl(row['dataset_path'])
    abst_ratio = sum(is_abstention_training_case(r) for r in rows) / len(rows) if rows else 0.0
    quick_results.append({**row, 'quick_train_abstention_ratio': abst_ratio})

write_csv(quick_results, ABLATION_ROOT / 'ablation_quick_results.csv')
write_json({'quick_results': quick_results}, ARTIFACT_ROOT / '06_ablation_quick_results.json')
print(json.dumps(quick_results, indent=2, ensure_ascii=False))
